# Byzantine-Robust Aggregation: Poisoning Defense

Federated learning is vulnerable to poisoning attacks: malicious clients send corrupted updates to the server. Byzantine-robust aggregation algorithms detect and filter out these attacks.

## Poisoning Attack Taxonomy

**Model poisoning**: a malicious client modifies its local model update to degrade global model accuracy or introduce a backdoor.

**Targeted poisoning (backdoor)**: the attack causes the global model to misclassify specific inputs while maintaining normal accuracy on other inputs. E.g., a model that correctly classifies all images but misclassifies images with a small trigger pattern.

**Untargeted poisoning**: degrades overall model accuracy by sending random or adversarially crafted updates.

**Sybil attack**: a single adversary controls multiple client identities to amplify their influence on aggregation.

In [ ]:
import math, random
from dataclasses import dataclass

@dataclass
class ClientUpdate:
    client_id: int
    gradient: list
    n_samples: int
    is_malicious: bool = False

def fedavg_aggregate(updates: list) -> list:
    total = sum(u.n_samples for u in updates)
    n_params = len(updates[0].gradient)
    return [sum(u.gradient[i]*u.n_samples for u in updates)/total for i in range(n_params)]

def krum_aggregate(updates: list, n_byzantine: int = 1) -> list:
    grads = [u.gradient for u in updates]
    n = len(grads)
    n_select = n - n_byzantine - 2
    distances = [[0.0]*n for _ in range(n)]
    for i in range(n):
        for j in range(i+1, n):
            d = math.sqrt(sum((grads[i][k]-grads[j][k])**2 for k in range(len(grads[i]))))
            distances[i][j] = distances[j][i] = d
    scores = [sum(sorted(distances[i])[1:n_select+1]) for i in range(n)]
    best = scores.index(min(scores))
    return grads[best]

def trimmed_mean_aggregate(updates: list, trim_frac: float = 0.2) -> list:
    n = len(updates)
    n_trim = int(n * trim_frac)
    n_params = len(updates[0].gradient)
    result = []
    for i in range(n_params):
        vals = sorted(u.gradient[i] for u in updates)
        trimmed = vals[n_trim:n-n_trim] if n_trim > 0 else vals
        result.append(sum(trimmed)/len(trimmed))
    return result

rng = random.Random(42)

def gen_honest_update(client_id, true_grad, noise=0.1):
    grad = [true_grad + rng.gauss(0, noise)]
    return ClientUpdate(client_id, grad, rng.randint(100, 500))

def gen_malicious_update(client_id, attack_scale=10.0):
    grad = [attack_scale * rng.choice([-1, 1])]
    return ClientUpdate(client_id, grad, 100, is_malicious=True)

true_gradient = [1.0]
n_honest, n_malicious = 8, 2
updates = ([gen_honest_update(i, 1.0) for i in range(n_honest)] +
           [gen_malicious_update(n_honest+i, 10.0) for i in range(n_malicious)])

fa = fedavg_aggregate(updates)
krum = krum_aggregate(updates, n_byzantine=n_malicious)
tm = trimmed_mean_aggregate(updates, trim_frac=0.2)

print(f'True gradient: {true_gradient[0]}')
print(f'FedAvg (no defense):  {fa[0]:.4f} (error: {abs(fa[0]-1.0):.4f})')
print(f'Krum:                 {krum[0]:.4f} (error: {abs(krum[0]-1.0):.4f})')
print(f'Trimmed Mean (20%):   {tm[0]:.4f} (error: {abs(tm[0]-1.0):.4f})')

## FLTrust and Reputation-Based Aggregation

FLTrust (Cao et al. 2020) extends Byzantine robustness by using a small server-side clean dataset to compute a trust score for each client:
1. Server computes its own gradient on a small clean dataset
2. Each client's update is scored by its cosine similarity to the server gradient
3. Updates with negative similarity (pointing the wrong direction) are rejected
4. Aggregation is weighted by trust scores

FLTrust is more robust than Krum and Trimmed Mean in practical settings because it uses semantic information (direction alignment) rather than just geometric distance.